In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

torch.manual_seed(42)

In [2]:
# 读取文本文件
with open('tiny_corpus_rnn.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# 构建字符表
vocab = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(vocab)}   # 字符 -> 索引
itos = {i: ch for ch, i in stoi.items()}       # 索引 -> 字符
vocab_size = len(vocab)

# 将全文编码为整数序列
ids = [stoi[ch] for ch in text]

print(f"文本总长度: {len(text)}")
print(f"字符表大小: {vocab_size}")
print(f"前200字符预览: {text[:200]}")

文本总长度: 165488
字符表大小: 291
前200字符预览: 人工智能是一门研究如何让机器表现出智能行为的学科。 循环神经网络擅长处理序列，因为它用隐藏状态记住上下文信息。 In practice, 请记录每50步的loss，并在训练前后各生成一段文本进行对比。（主题：特征工程） However, 正则化可以缓解过拟合，例如L2权重衰减、Dropout与数据增强。（主题：超参数搜索）
评估模型时需要区分训练集与验证集，避免把测试集当作调参工具。 人工智能是一


In [3]:
T = 32   # 每个样本的时间步长度

def get_data(ids, T):
    """生成输入序列 x 和目标序列 y (每个样本长度 T)"""
    xs, ys = [], []
    for i in range(0, len(ids) - T, 1):   # 滑动窗口步长=1
        xs.append(ids[i:i+T])
        ys.append(ids[i+1:i+T+1])
    return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)

X, Y = get_data(ids, T)
print(f"总样本数: {X.shape[0]}, 每个样本长度: {T}")

总样本数: 165456, 每个样本长度: 32


In [4]:
class CharDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

batch_size = 64
dataset = CharDataset(X, Y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [5]:
class MyRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        # 参数: Wxh (input_size, hidden_size), Whh (hidden_size, hidden_size), bh (hidden_size)
        self.Wxh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
        self.Whh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.bh = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x_t, h_prev):
        # x_t: (B, input_size)
        # h_prev: (B, hidden_size)
        # 输出: h_t = tanh(x_t @ Wxh + h_prev @ Whh + bh)
        h_t = torch.tanh(x_t @ self.Wxh + h_prev @ self.Whh + self.bh)
        return h_t

In [6]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.cell = MyRNNCell(embed_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h0=None):
        # x: (B, T)
        B, T = x.shape
        # 嵌入: (B, T, embed_size)
        x_emb = self.embedding(x)

        # 初始化隐藏状态
        if h0 is None:
            h = torch.zeros(B, self.cell.hidden_size, device=x.device)
        else:
            h = h0

        h_seq = []
        for t in range(T):
            h = self.cell(x_emb[:, t, :], h)   # (B, hidden_size)
            h_seq.append(h)
        # 堆叠所有时间步: (B, T, hidden_size)
        h_seq = torch.stack(h_seq, dim=1)

        # 输出层: (B, T, vocab_size)
        logits = self.fc(h_seq)
        return logits, h   # 返回最后一个隐藏状态用于生成

    def init_hidden(self, batch_size, device):
        return torch.zeros(batch_size, self.cell.hidden_size, device=device)

In [7]:
# 超参数
embed_size = 32
hidden_size = 128
learning_rate = 1e-3
num_epochs = 10   # 可适当增加

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CharRNN(vocab_size, embed_size, hidden_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("开始训练...")
for epoch in range(num_epochs):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)  # (B, T), (B, T)
        # 前向
        logits, _ = model(batch_x)
        # 计算损失: reshape to (B*T, V) 和 (B*T)
        loss = criterion(logits.view(-1, vocab_size), batch_y.view(-1))
        # 反向
        optimizer.zero_grad()
        loss.backward()
        # 梯度裁剪（防止爆炸）
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1:2d}/{num_epochs}, Loss = {avg_loss:.4f}")

开始训练...
Epoch  1/10, Loss = 0.5246
Epoch  2/10, Loss = 0.1795
Epoch  3/10, Loss = 0.1758
Epoch  4/10, Loss = 0.1734
Epoch  5/10, Loss = 0.1715
Epoch  6/10, Loss = 0.1698
Epoch  7/10, Loss = 0.1685
Epoch  8/10, Loss = 0.1674
Epoch  9/10, Loss = 0.1665
Epoch 10/10, Loss = 0.1658


In [8]:
def sample(model, start_str, gen_len=200, temperature=1.0, device='cpu'):
    model.eval()
    # 将起始字符串转为索引
    input_ids = torch.tensor([stoi[ch] for ch in start_str], dtype=torch.long).unsqueeze(0).to(device)  # (1, len_start)
    # 初始化隐藏状态
    h = model.init_hidden(1, device)
    # 先处理起始字符串，更新隐藏状态
    with torch.no_grad():
        for t in range(input_ids.size(1)):
            x_t = input_ids[:, t:t+1]          # (1, 1)
            x_emb = model.embedding(x_t)       # (1, 1, embed_size)
            h = model.cell(x_emb.squeeze(1), h) # (1, hidden_size)
        # 当前最后一个字符的索引（用于生成下一个）
        curr_id = input_ids[:, -1].item()

    generated = list(start_str)   # 存储生成的字符
    for _ in range(gen_len):
        # 当前字符的嵌入
        x_curr = torch.tensor([[curr_id]], dtype=torch.long, device=device)   # (1, 1)
        x_emb = model.embedding(x_curr)       # (1, 1, embed_size)
        h = model.cell(x_emb.squeeze(1), h)   # (1, hidden_size)
        logits = model.fc(h)                  # (1, vocab_size)
        # 温度采样
        probs = F.softmax(logits / temperature, dim=-1).squeeze(0)
        next_id = torch.multinomial(probs, 1).item()
        generated.append(itos[next_id])
        curr_id = next_id
    return ''.join(generated)

In [9]:
# 训练前（随机初始化）
print("训练前生成（200字符）:")
print(sample(model, start_str="人工", gen_len=200, temperature=1.0, device=device))

# 训练后
print("\n训练后生成（200字符）:")
print(sample(model, start_str="人工", gen_len=200, temperature=0.8, device=device))

训练前生成（200字符）:
人工。
换句话说，在训练过程中，学习率决定了参数更新的步幅，过大可能震荡，过小可能收敛太慢。（主题：部署与监控） 实验的重点是理解张量形状与训练循环，而不是追求最强性能。 卷积网络擅长处理图像，因为它利用局部连接与权值共享。 However, For character-level language modeling, the task is to predict the next character

训练后生成（200字符）:
人工下文信息。（主题：可解释性） 请记录每50步的loss，并在训练前后各生成一段文本进行对比。 在监督学习中，模型通过最小化损失来拟合输入与标签之间的映射。 循环神经网络擅长处理序列，因为它用隐藏状态记住上下文信息。 在训练过程中，学习率决定了参数更新的步幅，过大可能震荡，过小可能收敛太慢。 Meanwhile, 正则化可以缓解过拟合，例如L2权重衰减、Dropout与数据增强。
For chara
